# Step 8: Deploy & Submission (Ứng Dụng & Nộp Bài Kaggle)
### Theo Quy Trình 8 Bước Từ Slide Bài Giảng Chapter 4 (NEU FIT)

---

### 1. Phân Tích Chuyên Sâu Cách Làm:
Slide 41 & 58 đặt yêu cầu: *Output of this step: someone who does not code can use the model.*
Có 3 cách triển khai chính:
1. **Model Persistence (`joblib.dump` / `joblib.load`):** Lưu trữ mô hình vào ổ đĩa để tái sử dụng mà không cần huấn luyện lại.
2. **Kaggle Submission File (Slide 56):** Xuất file `submission.csv` với 3.869 dòng chứa cột `id` và doanh thu dự báo `revenue`.
3. **Interactive What-If Scenario Form (Slide 42 & 58):** Xây dựng biểu mẫu nhập liệu cho nhân viên/quản lý chuỗi cà phê:
   - Người quản lý chọn cửa hàng, ngày kinh doanh, có chạy khuyến mãi hay không, nhiệt độ thời tiết.
   - Hệ thống lập tức trả về: **Doanh thu dự báo** và khuyến nghị hành động kinh doanh!

### 2. Dùng Những Gì Để Làm?
- **Thư viện:** `joblib`, `pandas`, `numpy`, cú pháp `@param` tương thích giao diện Google Colab & VS Code Notebook.

In [1]:
import os, sys
import pandas as pd
import numpy as np
import joblib

for p in [os.path.abspath(".."), os.path.abspath("."), os.path.abspath("../..")]:
    if os.path.exists(os.path.join(p, "data")):
        DATA_DIR = os.path.join(p, "data")
    if p not in sys.path and os.path.exists(os.path.join(p, "src")):
        sys.path.insert(0, p)

from src.config import CAFE_MODEL_V3_PATH, SUBMISSION_V3_PATH, SUBMISSION_PATH
from src.cleaning import clean_stores, clean_test
from src.cafe_model_v3 import CafeRevenueModelV3

# 1. Tải mô hình đã lưu từ models/
print(f"Đang tải mô hình từ: {CAFE_MODEL_V3_PATH}")
model = CafeRevenueModelV3.load(CAFE_MODEL_V3_PATH)
print(">>> ĐÃ NẠP MÔ HÌNH THÀNH CÔNG TỪ JOBLIB!")

Đang tải mô hình từ: E:\study\AI\test 1\Kaggle_Competition\data\cafe\models\cafe_revenue_model_v3.pkl
>>> ĐÃ NẠP MÔ HÌNH THÀNH CÔNG TỪ JOBLIB!


### 2. Sinh File Nộp Bài Chuẩn Định Dạng Kaggle (3.869 dòng)

In [2]:
# Đọc test.csv và dự báo
stores_c = clean_stores(pd.read_csv(os.path.join(DATA_DIR, "stores.csv")))
test_c, _ = clean_test(pd.read_csv(os.path.join(DATA_DIR, "test.csv")), stores_c)

test_preds = model.predict(test_c)

sub_df = pd.DataFrame({
    "id": test_c["id"].values,
    "revenue": np.round(test_preds, 4)
})

# Lưu submission
sub_df.to_csv(SUBMISSION_V3_PATH, index=False)
sub_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Đã lưu file nộp bài vào: {SUBMISSION_V3_PATH}")
print(f"Số dòng: {len(sub_df)} | Không có NaN: {sub_df['revenue'].isna().sum() == 0}")
display(sub_df.head(5))

Đã lưu file nộp bài vào: E:\study\AI\test 1\Kaggle_Competition\data\cafe\submissions\submission_v3.csv
Số dòng: 3869 | Không có NaN: True


,id,revenue
0,100000,6.5602
1,100001,8.7557
2,100002,21.4346
3,100003,33.1717
4,100004,9.5288


### 3. Biểu Mẫu Nhập Liệu Tương Tác What-If Cho Quản Lý Cửa Hàng (Interactive Form)
Dựa theo Slide 42 & 58: Cung cấp giao diện mô phỏng "What-If" cho người dùng nghiệp vụ.

In [3]:
# @title BIỂU MẪU DỰ BÁO DOANH THU & KỊCH BẢN WHAT-IF { run: "auto" }
store_id = "S01" # @param ["S01", "S03", "S04", "S05", "S10", "S20", "S45"]
date_input = "2026-02-14" # @param {type:"string"}
is_promo = 1 # @param [0, 1] {type:"integer"}

sample_query = pd.DataFrame([{
    "date": pd.to_datetime(date_input),
    "store_id": store_id,
    "is_promo": is_promo
}])

predicted_val = model.predict(sample_query)[0]

# Kịch bản đối chứng: Nếu không chạy khuyến mãi
sample_no_promo = sample_query.copy()
sample_no_promo["is_promo"] = 0
base_val = model.predict(sample_no_promo)[0]

print("=" * 65)
print(f"KẾT QUẢ DỰ BÁO DOANH THU CHO CỬA HÀNG: {store_id} NGÀY {date_input}")
print("=" * 65)
print(f"- Doanh thu dự kiến                  : {predicted_val:.2f} triệu VNĐ")
print(f"- Doanh thu ngày thường (Không Promo): {base_val:.2f} triệu VNĐ")
if is_promo == 1:
    diff = predicted_val - base_val
    print(f"- Chênh lệch do Khuyến Mãi           : +{diff:.2f} triệu VNĐ (+{diff/base_val*100:.1f}%)")
    print(f"- Khuyến nghị vận hành               : Cần chuẩn bị thêm 30% nguyên vật liệu và nhân sự pha chế.")
else:
    print(f"- Khuyến nghị vận hành               : Duy trì định mức ca trực tiêu chuẩn.")
print("=" * 65)

KẾT QUẢ DỰ BÁO DOANH THU CHO CỬA HÀNG: S01 NGÀY 2026-02-14
- Doanh thu dự kiến                  : 7.20 triệu VNĐ
- Doanh thu ngày thường (Không Promo): 5.47 triệu VNĐ
- Chênh lệch do Khuyến Mãi           : +1.72 triệu VNĐ (+31.5%)
- Khuyến nghị vận hành               : Cần chuẩn bị thêm 30% nguyên vật liệu và nhân sự pha chế.
